In [1]:
from pathlib import Path
import kagglehub
import os
import shutil

/home/chiragg21/dev_projects/ml_learning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
base_dir = Path("/home/chiragg21/dev_projects/ml_learning/")
dest_path = base_dir / "data" / "tabular_data" / "raw"

if (dest_path / "adult.csv").exists():
    print("File already exists, skipping download.")
else:
    dest_path.mkdir(parents=True, exist_ok=True)
    cache_path = Path(kagglehub.dataset_download("uciml/adult-census-income")) / 'adult.csv'
    shutil.move(cache_path, dest_path)

File already exists, skipping download.


In [4]:
import polars as pl

data_path = dest_path / 'adult.csv'
raw_data = pl.read_csv(data_path)
raw_data.head(5)

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str
90,"""?""",77053,"""HS-grad""",9,"""Widowed""","""?""","""Not-in-family""","""White""","""Female""",0,4356,40,"""United-States""","""<=50K"""
82,"""Private""",132870,"""HS-grad""",9,"""Widowed""","""Exec-managerial""","""Not-in-family""","""White""","""Female""",0,4356,18,"""United-States""","""<=50K"""
66,"""?""",186061,"""Some-college""",10,"""Widowed""","""?""","""Unmarried""","""Black""","""Female""",0,4356,40,"""United-States""","""<=50K"""
54,"""Private""",140359,"""7th-8th""",4,"""Divorced""","""Machine-op-inspct""","""Unmarried""","""White""","""Female""",0,3900,40,"""United-States""","""<=50K"""
41,"""Private""",264663,"""Some-college""",10,"""Separated""","""Prof-specialty""","""Own-child""","""White""","""Female""",0,3900,40,"""United-States""","""<=50K"""


Dataset structure and Schema

In [15]:
print("Shape:", raw_data.shape)

Shape: (32561, 15)


In [35]:
print("Schema:\n")
for col, dtype in raw_data.schema.items():
    print(f"{col}: {dtype}")

Schema:

age: Int64
workclass: String
fnlwgt: Int64
education: String
education.num: Int64
marital.status: String
occupation: String
relationship: String
race: String
sex: String
capital.gain: Int64
capital.loss: Int64
hours.per.week: Int64
native.country: String
income: String


Missing Values

In [40]:
raw_data.null_count()

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [54]:
dirty_placeholders = ['?', 'NA', 'N/A', 'na', 'n/a', 'null', 'NULL', 'None', 'none']

exprs = []

for col, type in raw_data.schema.items():
    condition = pl.col(col).is_null() 
    if type == pl.String:
        condition |= pl.col(col).is_in(dirty_placeholders)
    if type.is_numeric():
        condition |= pl.col(col).is_nan()


    exprs.append(pl.col(col).filter(condition).len())
x = (raw_data.select(exprs))

cols_with_missing_values = {}
for col, type in raw_data.schema.items():
    if x[col][0]>0:
        cols_with_missing_values[col] = {'type': type, 'missing_count': x[col][0]}

cols_with_missing_values

{'workclass': {'type': String, 'missing_count': 1836},
 'occupation': {'type': String, 'missing_count': 1843},
 'native.country': {'type': String, 'missing_count': 583}}

In [60]:
import polars.selectors as cs

numerical_columns = [col for col, type in raw_data.schema.items() if type.is_numeric()]
categorical_columns = [col for col, type in raw_data.schema.items() if type==pl.String]

print(f"numercal columns: {numerical_columns}\ncategorical columns: {categorical_columns}")

numercal columns: ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
categorical columns: ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country', 'income']


In [63]:
raw_data.select(
    [
        pl.col(column)
        .eq('?')
        .mean()
        .mul(100)
        .alias(column)
        for column in categorical_columns
    ]
)

workclass,education,marital.status,occupation,relationship,race,sex,native.country,income
f64,f64,f64,f64,f64,f64,f64,f64,f64
5.638647,0.0,0.0,5.660146,0.0,0.0,0.0,1.790486,0.0


In [64]:
raw_data.select(
    [
        pl.col(column).n_unique().alias(column)
        for column in raw_data.columns
    ]
)

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
73,9,21648,16,16,7,15,6,5,2,119,92,94,42,2


In [71]:
raw_data["workclass"].value_counts(normalize=True, sort=True)

workclass,proportion
str,f64
"""Private""",0.69703
"""Self-emp-not-inc""",0.078038
"""Local-gov""",0.064279
"""?""",0.056386
"""State-gov""",0.039864
"""Self-emp-inc""",0.034274
"""Federal-gov""",0.029483
"""Without-pay""",0.00043
"""Never-worked""",0.000215


In [72]:
for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(raw_data[column].unique())


--- workclass ---
shape: (9,)
Series: 'workclass' [str]
[
	"Private"
	"Self-emp-not-inc"
	"Local-gov"
	"State-gov"
	"?"
	"Never-worked"
	"Without-pay"
	"Self-emp-inc"
	"Federal-gov"
]

--- education ---
shape: (16,)
Series: 'education' [str]
[
	"Doctorate"
	"Assoc-voc"
	"10th"
	"11th"
	"1st-4th"
	…
	"Some-college"
	"12th"
	"Bachelors"
	"HS-grad"
	"7th-8th"
]

--- marital.status ---
shape: (7,)
Series: 'marital.status' [str]
[
	"Married-spouse-absent"
	"Married-civ-spouse"
	"Widowed"
	"Never-married"
	"Divorced"
	"Separated"
	"Married-AF-spouse"
]

--- occupation ---
shape: (15,)
Series: 'occupation' [str]
[
	"?"
	"Priv-house-serv"
	"Armed-Forces"
	"Protective-serv"
	"Prof-specialty"
	…
	"Transport-moving"
	"Adm-clerical"
	"Sales"
	"Other-service"
	"Machine-op-inspct"
]

--- relationship ---
shape: (6,)
Series: 'relationship' [str]
[
	"Own-child"
	"Husband"
	"Wife"
	"Other-relative"
	"Not-in-family"
	"Unmarried"
]

--- race ---
shape: (5,)
Series: 'race' [str]
[
	"Amer-Indian-Eskimo"
	

In [73]:
raw_data.select(
    numerical_columns
).describe()

statistic,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
str,f64,f64,f64,f64,f64,f64
"""count""",32561.0,32561.0,32561.0,32561.0,32561.0,32561.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",38.581647,189778.366512,10.080679,1077.648844,87.30383,40.437456
"""std""",13.640433,105549.977697,2.57272,7385.292085,402.960219,12.347429
"""min""",17.0,12285.0,1.0,0.0,0.0,1.0
"""25%""",28.0,117827.0,9.0,0.0,0.0,40.0
"""50%""",37.0,178356.0,10.0,0.0,0.0,40.0
"""75%""",48.0,237051.0,12.0,0.0,0.0,45.0
"""max""",90.0,1.484705e6,16.0,99999.0,4356.0,99.0


In [77]:
raw_data.select(
    [
        (
            pl.col(column).max() - pl.col(column).min()
        ).alias(f"{column}_range")
        for column in numerical_columns
    ]
)

age_range,fnlwgt_range,education.num_range,capital.gain_range,capital.loss_range,hours.per.week_range
i64,i64,i64,i64,i64,i64
73,1472420,15,99999,4356,98


In [78]:
raw_data.is_duplicated().sum()

47

In [82]:
raw_data.select(
    [
        pl.col(column)
        .str.strip_chars()
        .n_unique()
        .alias(column)
        for column in categorical_columns
    ]
)

workclass,education,marital.status,occupation,relationship,race,sex,native.country,income
u32,u32,u32,u32,u32,u32,u32,u32,u32
9,16,7,15,6,5,2,42,2


In [84]:
cardinality = raw_data.select(
    [
        pl.col(column)
        .n_unique()
        .alias(column)
        for column in raw_data.columns
    ]
)
cardinality

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
73,9,21648,16,16,7,15,6,5,2,119,92,94,42,2


In [85]:
raw_data.select(
    [
        pl.col(column)
        .skew()
        .alias(column)
        for column in numerical_columns
    ]
)

age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
f64,f64,f64,f64,f64,f64
0.558718,1.446913,-0.311662,11.953297,4.594417,0.227632


In [87]:
raw_data['income'].value_counts()

income,count
str,u32
""">50K""",7841
"""<=50K""",24720


In [88]:
raw_data.select(
    numerical_columns
).corr()

age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
f64,f64,f64,f64,f64,f64
1.0,-0.076646,0.036527,0.077674,0.057775,0.068756
-0.076646,1.0,-0.043195,0.000432,-0.010252,-0.018768
0.036527,-0.043195,1.0,0.12263,0.079923,0.148123
0.077674,0.000432,0.12263,1.0,-0.031615,0.078409
0.057775,-0.010252,0.079923,-0.031615,1.0,0.054256
0.068756,-0.018768,0.148123,0.078409,0.054256,1.0


In [90]:
raw_data = raw_data.with_columns(
    [
        pl.when(pl.col(col)=='?')
        .then(None)
        .otherwise(pl.col(col))
        .alias(col)
        for col in categorical_columns
    ]
)
raw_data.null_count()

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,1836,0,0,0,0,1843,0,0,0,0,0,0,583,0


In [93]:
cleaned_data = raw_data.with_columns(
    [
        pl.col(col)
        .str.strip_chars()
        .str.to_lowercase()
        for col in categorical_columns
    ]
)
de_dup_data = cleaned_data.unique(keep='first')
de_dup_data

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str
47,"""private""",33710,"""bachelors""",13,"""married-civ-spouse""","""exec-managerial""","""husband""","""white""","""male""",3103,0,60,"""united-states""",""">50k"""
29,"""private""",123677,"""hs-grad""",9,"""married-civ-spouse""","""machine-op-inspct""","""wife""","""asian-pac-islander""","""female""",0,0,40,"""laos""","""<=50k"""
20,"""private""",113307,"""7th-8th""",4,"""never-married""","""machine-op-inspct""","""other-relative""","""white""","""male""",0,0,40,"""united-states""","""<=50k"""
41,"""private""",125831,"""hs-grad""",9,"""married-civ-spouse""","""craft-repair""","""husband""","""white""","""male""",0,2051,60,"""united-states""","""<=50k"""
25,"""private""",81286,"""hs-grad""",9,"""married-civ-spouse""","""craft-repair""","""husband""","""white""","""male""",0,0,45,"""united-states""","""<=50k"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
23,"""private""",388811,"""hs-grad""",9,"""never-married""","""adm-clerical""","""own-child""","""white""","""female""",0,0,40,"""united-states""","""<=50k"""
55,"""local-gov""",223716,"""some-college""",10,"""divorced""","""exec-managerial""","""not-in-family""","""amer-indian-eskimo""","""female""",0,0,40,"""united-states""","""<=50k"""
50,"""private""",184424,"""some-college""",10,"""married-civ-spouse""","""adm-clerical""","""wife""","""white""","""female""",0,1902,38,"""united-states""",""">50k"""


In [101]:
(de_dup_data.group_by('education').agg(pl.col('education.num').n_unique().alias('num_count')))

education,num_count
str,u32
"""9th""",1
"""1st-4th""",1
"""10th""",1
"""11th""",1
"""masters""",1
…,…
"""assoc-acdm""",1
"""assoc-voc""",1
"""bachelors""",1


In [108]:
q1 = de_dup_data["age"].quantile(0.25)
q3 = de_dup_data["age"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(iqr)

20.0


In [111]:
outliers = de_dup_data.filter(
    (pl.col("age") < lower) |
    (pl.col("age") > upper)
)
outliers

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str
79,"""self-emp-not-inc""",158319,"""hs-grad""",9,"""widowed""","""other-service""","""not-in-family""","""white""","""female""",0,0,24,"""united-states""","""<=50k"""
81,null,106765,"""some-college""",10,"""widowed""",null,"""unmarried""","""white""","""female""",0,0,4,"""united-states""","""<=50k"""
88,"""self-emp-not-inc""",206291,"""prof-school""",15,"""married-civ-spouse""","""prof-specialty""","""husband""","""white""","""male""",0,0,40,"""united-states""","""<=50k"""
90,"""self-emp-not-inc""",82628,"""hs-grad""",9,"""never-married""","""exec-managerial""","""not-in-family""","""white""","""male""",2964,0,12,"""united-states""","""<=50k"""
79,"""self-emp-not-inc""",103684,"""hs-grad""",9,"""married-civ-spouse""","""farming-fishing""","""husband""","""white""","""male""",0,0,40,"""united-states""","""<=50k"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
79,null,165209,"""hs-grad""",9,"""married-civ-spouse""",null,"""husband""","""white""","""male""",0,0,40,"""united-states""","""<=50k"""
79,"""private""",266119,"""bachelors""",13,"""married-civ-spouse""","""sales""","""husband""","""white""","""male""",0,0,40,"""united-states""","""<=50k"""
83,null,251951,"""hs-grad""",9,"""widowed""",null,"""not-in-family""","""white""","""female""",0,0,20,"""united-states""","""<=50k"""


Skewness

In [113]:
skew_score = de_dup_data.select(
    [
        pl.col(column)
        .skew()
        .alias(column)
        for column in numerical_columns
    ]
)

In [120]:
skew_score.row(0, named=True).items()

dict_items([('age', 0.5576374421375202), ('fnlwgt', 1.4476361664196684), ('education.num', -0.30948599142388766), ('capital.gain', 11.94885194275666), ('capital.loss', 4.592490690725545), ('hours.per.week', 0.22874833162171856)])

In [124]:
skew_score

age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
f64,f64,f64,f64,f64,f64
0.557637,1.447636,-0.309486,11.948852,4.592491,0.228748


In [123]:
df = de_dup_data

skewed_cols = [k for k,v in skew_score.row(0, named=True).items() if v>1 or v<-1]
skewed_cols
df = df.with_columns(
    [pl.col(col)
     .log1p()
     .alias(col)
     for col in skewed_cols]
)

new_skew_score = df.select(
    [
        pl.col(col)
        .skew()
        .alias(col)
        for col in skewed_cols
    ]
)

new_skew_score

fnlwgt,capital.gain,capital.loss
f64,f64,f64
-0.842165,3.094524,4.305502
